# ContraMesh: The Algorithmic Legal Bodyguard
### Kaggle Dual T4 GPU Deployment Guide

This notebook demonstrates how to deploy **ContraMesh** on a Kaggle Notebook utilizing dual T4 GPUs (GPU T4 x2). We split Gemma's weights using **vLLM Tensor Parallelism** and run the FastAPI backend + Memgraph database concurrently.

## Step 1: Install Dependencies

We install `vllm` for LLM parallel execution, `z3-solver` for logic calculations, `neo4j` for Memgraph database connectivity, and backend routing libraries.

In [ ]:
# Install vLLM, Z3 Solver, and backend drivers
!pip install -q vllm z3-solver neo4j pypdf fastapi uvicorn nest-asyncio

## Step 2: Download & Extract Memgraph DB Binary

Since we don't have root Docker permissions on Kaggle, we run the Memgraph binary directly as a user process in the background.

In [ ]:
# Download and unpack Memgraph executable
!wget -q https://download.memgraph.com/memgraph/v2.14.0/debian-12/memgraph_2.14.0-1_amd64.deb
!dpkg -x memgraph_2.14.0-1_amd64.deb ./memgraph_temp

# Start Memgraph server in the background
import subprocess
import time

print("Launching Memgraph Graph Database server...")
memgraph_process = subprocess.Popen(
    ["./memgraph_temp/usr/bin/memgraph", "--bolt-port=7687", "--log-level=WARNING"],
    stdout=subprocess.PIPE,
    stderr=subprocess.PIPE
)
time.sleep(3)
print("Memgraph database is active on port 7687!")

## Step 3: Launch Gemma 4 model with vLLM Tensor Parallelism

We boot the vLLM OpenAI-compatible server utilizing **both T4 GPUs** by specifying `--tensor-parallel-size 2`.

In [ ]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0,1"

print("Launching vLLM server on Dual T4 GPUs (Tensor Parallel Size = 2)...")
vllm_command = "python -m vllm.entrypoints.openai.api_server --model google/gemma-2-9b-it --tensor-parallel-size 2 --port 8000 --disable-log-requests"
vllm_process = subprocess.Popen(vllm_command, shell=True)
print("vLLM server started in background. Loading model weights (takes ~3 mins)...")

## Step 4: Start ContraMesh FastAPI Backend Router

Write the API server script dynamically and launch it on port `8001`.

In [ ]:
# Start the API server
print("Launching ContraMesh FastAPI server...")
backend_cmd = "python main.py"
backend_process = subprocess.Popen(backend_cmd, shell=True, env={**os.environ, "PORT": "8001", "VLLM_API_URL": "http://localhost:8000/v1"})
time.sleep(3)
print("ContraMesh API is ready at http://localhost:8001!")

## Step 5: Expose API Server via localtunnel

Expose port 8001 to get a public URL that you can paste into the local React app.

In [ ]:
# Install localtunnel for public URL exposure
!npm install -g localtunnel

print("Requesting public URL for frontend connection...")
!lt --port 8001